# Sudoku Knowledge Representation & Inference

**Group 1:** Aman Vijay Khanna, Harn Kai Jun Clement, Kruthi Shiva Kumar, Shah Pratik

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
    solve_full_grid_fc_one_pass
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])


5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
# Attempt (a): solve the puzzle with pl_resolution / tt_entails on the general KB.
#
# The instructions say to uncomment this and hit Kernel > Interrupt after ~30s.
# We run it under a guard instead, so the notebook can be submitted with its
# output visible. The guard does two things on top of a timer:
#
#   * pl_resolution builds its whole list of clause pairs before resolving any
#     of them, so it eats memory fast (close to 1 GB within a couple of seconds
#     on this KB). So it gets a memory limit as well as a time limit.
#     tt_entails doesn't need one. It fails on time, not space, and giving
#     it one would only hide how it fails.
#   * A Python thread can't be cancelled from outside, so a thread we stopped
#     waiting on would keep using memory while the rest of the notebook runs.
#     We raise an exception inside it to actually stop it, and keep raising
#     it until the thread is gone, because a single raise can get lost.

import os
import sys
import time
import ctypes
import threading

try:
    import psutil
except ImportError:
    psutil = None

TIME_BUDGET = 20     # seconds, roughly the ~30s the instructions suggest
MEMORY_BUDGET = 800  # extra MB pl_resolution may allocate before we stop it


class GaveUp(BaseException):
    """Raised inside a worker thread we have stopped waiting on."""


# A stopped thread unwinds with GaveUp, which Python otherwise reports as a long
# traceback. Silence just that one exception so the output stays readable.
_thread_excepthook, _unraisable_hook = threading.excepthook, sys.unraisablehook


def _quiet(args, fallback):
    if args.exc_type is not GaveUp:
        fallback(args)


threading.excepthook = lambda a: _quiet(a, _thread_excepthook)
sys.unraisablehook = lambda a: _quiet(a, _unraisable_hook)


def try_inference(label, fn, time_limit=TIME_BUDGET, mem_limit=None):
    """Run fn() on a background thread and report what happened.

    Stops at whichever comes first: a returned value, an exception, the time
    budget, or the memory budget if one was given. Returns a one-line summary.
    """
    out = {}

    def target():
        try:
            out['value'] = fn()
        except Exception as e:      # the algorithm itself raised an error
            out['error'] = f'{type(e).__name__}: {e}'

    def give_up(reason):
        # One raise isn't always enough: we saw it get lost and the thread keep
        # running (and allocating) for the rest of the notebook. So we keep
        # raising it until the thread has really stopped.
        deadline = time.time() + 30
        while thread.is_alive() and time.time() < deadline:
            ctypes.pythonapi.PyThreadState_SetAsyncExc(
                ctypes.c_ulong(thread.ident), ctypes.py_object(GaveUp))
            thread.join(0.5)
        if thread.is_alive():
            reason += ', and the thread would not stop'
        return f'{label}: gave up, {reason}'

    watch_memory = psutil is not None and mem_limit is not None
    proc = psutil.Process(os.getpid()) if watch_memory else None

    def memory_mb():
        # Private memory where psutil has it (Windows), because the working set
        # (rss) shrinks when Windows pages memory out. rss everywhere else.
        info = proc.memory_info()
        return getattr(info, 'private', info.rss) / 1e6

    base = memory_mb() if watch_memory else 0

    thread = threading.Thread(target=target, daemon=True)
    start = time.time()
    thread.start()

    while thread.is_alive():
        thread.join(0.5)
        elapsed = time.time() - start
        if watch_memory:
            used = memory_mb() - base
            if used > mem_limit:
                return give_up(f'had allocated {used:.0f} MB after {elapsed:.0f}s')
        if elapsed > time_limit:
            return give_up(f'still running after {time_limit}s')

    elapsed = time.time() - start
    if 'error' in out:
        return f'{label}: raised {out["error"]} after {elapsed:.1f}s'
    return f'{label}: returned {out["value"]} in {elapsed:.2f}s'


# How big a problem are we handing these two algorithms?
general_kb = build_general_kb(n, box_h, box_w, givens)
symbols = set()
for clause in general_kb.clauses:
    symbols |= prop_symbols(clause)

num_clauses = len(general_kb.clauses)
num_pairs = num_clauses * (num_clauses - 1) // 2
print(f'general KB: {num_clauses} clauses over {len(symbols)} symbols')
print(f'  tt_entails enumerates 2^{len(symbols)} models')
print(f'  pl_resolution compares {num_pairs:,} clause pairs in round 1 alone')
print()

# Cell (1,1) is empty in this puzzle and its answer is 1, so Is1_1_1 is
# entailed. A correct algorithm has to return True here.
r, c, v = 1, 1, givens.get((1, 1), 1)
query = atom('Is', r, c, v)
print(f'query: {query}   (cell ({r},{c}) is empty; expected answer True)')

if psutil:
    print(try_inference('pl_resolution', lambda: pl_resolution(general_kb, query),
                        mem_limit=MEMORY_BUDGET))
else:
    print('pl_resolution: skipped, needs psutil for the memory guard (pip install psutil)')
print(try_inference('tt_entails', lambda: tt_entails(associate('&', general_kb.clauses), query)))

general KB: 11775 clauses over 729 symbols
  tt_entails enumerates 2^729 models
  pl_resolution compares 69,319,425 clause pairs in round 1 alone

query: Is1_1_1   (cell (1,1) is empty; expected answer True)


pl_resolution: gave up, had allocated 978 MB after 5s


tt_entails: gave up, still running after 20s


In [6]:
# Same build_general_kb and the same two algorithms, just on smaller boards.
# If both answer correctly on a small board, the encoding is fine and the only
# problem on the 9x9 is size.

# 2x2 board, 1x2 boxes. Solution is 1 2 / 2 1, so Is2_2_1 is entailed.
tiny_givens = {(1, 1): 1}

# 4x4 board, 2x2 boxes. Solution is 1234 / 3412 / 2143 / 4321. We blank out the
# four cells on the diagonal, so Is1_1_1 is entailed but has to be inferred.
small_solution = {
    (1, 1): 1, (1, 2): 2, (1, 3): 3, (1, 4): 4,
    (2, 1): 3, (2, 2): 4, (2, 3): 1, (2, 4): 2,
    (3, 1): 2, (3, 2): 1, (3, 3): 4, (3, 4): 3,
    (4, 1): 4, (4, 2): 3, (4, 3): 2, (4, 4): 1,
}
small_givens = {rc: val for rc, val in small_solution.items() if rc[0] != rc[1]}

boards = [
    ('2x2 board (1x2 boxes)', 2, 1, 2, tiny_givens, atom('Is', 2, 2, 1)),
    ('4x4 board (2x2 boxes)', 4, 2, 2, small_givens, atom('Is', 1, 1, 1)),
]

for label, size, bh, bw, board_givens, board_query in boards:
    kb = build_general_kb(size, bh, bw, board_givens)
    syms = set()
    for clause in kb.clauses:
        syms |= prop_symbols(clause)
    print(f'{label}: {len(kb.clauses)} clauses, {len(syms)} symbols, 2^{len(syms)} models')
    print('  ' + try_inference('pl_resolution', lambda: pl_resolution(kb, board_query),
                               mem_limit=MEMORY_BUDGET))
    print('  ' + try_inference('tt_entails',
                               lambda: tt_entails(associate('&', kb.clauses), board_query)))
    print()

# Soundness check on the 2x2: a wrong value has to come back False, not just
# "no answer". If this printed True the encoding would be the problem, not size.
tiny_kb = build_general_kb(2, 1, 2, tiny_givens)
print(try_inference('pl_resolution on Is2_2_2 (wrong value)',
                    lambda: pl_resolution(tiny_kb, atom('Is', 2, 2, 2)),
                    mem_limit=MEMORY_BUDGET))

2x2 board (1x2 boxes): 21 clauses, 8 symbols, 2^8 models
  pl_resolution: returned True in 0.01s
  tt_entails: returned True in 0.00s

4x4 board (2x2 boxes): 412 clauses, 64 symbols, 2^64 models


  pl_resolution: gave up, still running after 20s


  tt_entails: gave up, still running after 20s

pl_resolution on Is2_2_2 (wrong value): returned False in 0.05s


### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


We ran both algorithms on the KB from `build_general_kb`. Neither could solve even one cell of the 9x9 puzzle.

On the real puzzle, the general KB has 11,775 clauses over 729 symbols. We asked about `Is1_1_1`. Cell (1,1) is empty and its value in the solution is 1, so the right answer is `True`, and it has to be worked out rather than looked up. That's also just one query. Solving the whole puzzle this way means up to 81 x 9 = 729 of them. Neither algorithm finished even this first one.

- `pl_resolution` never finished. It ran out of memory before it ran out of time. It builds the full list of clause pairs before resolving any of them (the `pairs = [(clauses[i], clauses[j]) ...]` line in `logic_.py`). With 11,775 clauses that's about 69 million pairs in the first round alone. It went past our 800 MB limit within a few seconds, still in round one. That's why the cell above watches memory as well as time, and stops the thread instead of just timing out and leaving it running. Left alone, it just keeps eating RAM.
- `tt_entails` never finished either. It has to check $2^{729}$ models, which is about $10^{219}$. It also recurses once per symbol, so it needs 729 nested calls just to reach one complete model. It was still running when we stopped it at 20 seconds.

Why the cost blows up for each:

- Resolution: one round compares every pair of clauses, so it's $O(N^2)$ for $N$ clauses. Every new resolvent gets added back, so $N$ grows and the next round costs even more.
- Model checking: there are $2^s$ models for $s$ symbols, so every extra symbol doubles the work. Each of the 81 cells adds 9 symbols.

To see where it breaks, we ran the same code on smaller boards:

| board | clauses | symbols | `pl_resolution` | `tt_entails` |
|---|---|---|---|---|
| 2x2 (1x2 boxes) | 21 | 8 | `True`, under a second | `True`, under a second |
| 4x4 (2x2 boxes) | 412 | 64 | no answer in 20 s | no answer in 20 s |
| 9x9 (3x3 boxes) | 11,775 | 729 | no answer (hit the 800 MB limit) | no answer in 20 s |

On the 2x2 both give the right answer, and resolution correctly says `False` for a wrong value (`Is2_2_2`). So the KB is fine, and both algorithms do what they promise. The only problem is size. It already breaks at 4x4, a board with just 412 clauses, and the real puzzle is about 29 times bigger than that.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [7]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [8]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.
#
import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)

# Completeness: pl_bc_entails must find the correct value for every cell.
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True

# Soundness: pl_bc_entails must not also confirm any incorrect value.
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
print(f"solve_full_grid_bc: {bc_time:.2f}s")


# Add in code to time solve_full_grid_fc_one_pass
t0 = time.time()
solved = solve_full_grid_fc_one_pass(n, box_h, box_w, givens)
fc_op_time = time.time() - t0
assert solved == puzzle['solution']

print(f"solve_full_grid_fc_one_pass: {fc_op_time:.2f}s")

solve_full_grid_fc: 32.77s
solve_full_grid_bc: 64.45s


solve_full_grid_fc_one_pass: 22.41s


Below is the grid `solve_full_grid_fc` reconstructs for this puzzle, next to the puzzle it started from, as (b) asks.

One note on (b). `solve_full_grid_fc` doesn't call the library's `pl_fc_entails` directly. It calls `pl_fc_entails_optimized` from `sudoku_solver.py`. That's the same forward chaining algorithm (Figure 7.15) with two changes: every fact it derives is added back into the KB, and a rule is removed once it has fired. So each new cell query starts from what the earlier ones already proved, instead of starting again from the givens. We made these changes to save time on the full grid. The answers are still checked against the solution for all 5 puzzles in the Q5 experiment further down.

In [9]:
# 2.3(b): display the grid solve_full_grid_fc reconstructs, next to the puzzle.
fc_grid = solve_full_grid_fc(n, box_h, box_w, givens)
assert fc_grid == puzzle['solution']


def grid_rows(values):
    rows = []
    for r in range(1, n + 1):
        digits = [str(values.get((r, c), '.')) for c in range(1, n + 1)]
        rows.append(' | '.join(' '.join(digits[i:i + box_w]) for i in range(0, n, box_w)))
        if r % box_h == 0 and r < n:
            rows.append('-' * len(rows[-1]))
    return rows


print(f"{'puzzle (givens only)':<21}    solved by solve_full_grid_fc")
for left, right in zip(grid_rows(givens), grid_rows(fc_grid)):
    print(f'{left:<21}    {right}')

puzzle (givens only)     solved by solve_full_grid_fc
. 3 . | . . . | . . .    1 3 6 | 9 8 5 | 4 7 2
2 . . | . . 7 | 8 6 .    2 9 4 | 3 1 7 | 8 6 5
5 8 . | 2 6 . | . 3 .    5 8 7 | 2 6 4 | 1 3 9
---------------------    ---------------------
7 5 . | . . . | . 8 .    7 5 1 | 4 9 6 | 2 8 3
. . . | . 7 . | 5 . 4    3 6 9 | 8 7 2 | 5 1 4
. . . | 5 3 . | . 9 6    4 2 8 | 5 3 1 | 7 9 6
---------------------    ---------------------
. 1 2 | . . 9 | . . .    8 1 2 | 6 4 9 | 3 5 7
6 4 . | . 5 8 | 9 . .    6 4 3 | 7 5 8 | 9 2 1
. . . | . 2 3 | . . .    9 7 5 | 1 2 3 | 6 4 8


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**

### 1(a) General KB strategy
General clauses can use negation directly, so we only need the $Is$ symbols. "Cell (r,c) is not v" is just $\neg Is_{r,c,v}$, so the $Not$ family isn't needed here. That keeps the KB smaller. The rules are encoded like this:

#### 1. At least 1 value per cell

Every cell needs at least one of its 9 values. So for each cell we OR all 9 symbols together. The clause is only true if at least one of them is true.

$$
Is_{1,1,1} \lor Is_{1,1,2} \lor Is_{1,1,3} \lor \cdots \lor Is_{1,1,8} \lor Is_{1,1,9}
$$

$$
Is_{1,2,1} \lor Is_{1,2,2} \lor Is_{1,2,3} \lor \cdots \lor Is_{1,2,8} \lor Is_{1,2,9}
$$

$$
\vdots
$$

This is repeated for every cell.


#### 2. At most one value per cell

A cell can't hold two values at once. We write this as implications: if $Is_{r,c,1}$ is true, none of $Is_{r,c,2}$ to $Is_{r,c,9}$ can be true. Same for every other value. The left side is the symbol for one value, and the right side is the AND of the negations of the other 8. For value 1 in cell (1,1):

$$
Is_{1,1,1} \Rightarrow \neg Is_{1,1,2} \land \neg Is_{1,1,3} \land \neg Is_{1,1,4} \land \neg Is_{1,1,5} \land \neg Is_{1,1,6} \land \neg Is_{1,1,7} \land \neg Is_{1,1,8} \land \neg Is_{1,1,9}
$$

An implication with an AND on the right splits into one implication per value:

$$
Is_{1,1,1} \Rightarrow \neg Is_{1,1,2}
$$

$$
Is_{1,1,1} \Rightarrow \neg Is_{1,1,3}
$$

$$
\vdots
$$

$$
Is_{1,1,1} \Rightarrow \neg Is_{1,1,9}
$$

This is repeated for every cell and every possible value it can take.

Each of these is really a 2-literal clause, since $A \Rightarrow \neg B$ is the same as $\neg A \lor \neg B$. `PropKB.tell` converts everything to CNF anyway. And one clause covers both directions, so each pair of values is only added once.

#### 3. Row uniqueness, Column uniqueness, Box uniqueness

Rows, columns and boxes work the same way as rule 2.

Row uniqueness - if a cell has value $v$, no other cell in its row can have $v$.

$$
Is_{1,1,1} \Rightarrow \neg Is_{1,2,1}, \quad Is_{1,1,1} \Rightarrow \neg Is_{1,3,1}, \quad \ldots \quad Is_{1,1,1} \Rightarrow \neg Is_{1,9,1}
$$

$$
Is_{1,1,2} \Rightarrow \neg Is_{1,2,2}, \quad Is_{1,1,2} \Rightarrow \neg Is_{1,3,2}, \quad \ldots \quad Is_{1,1,2} \Rightarrow \neg Is_{1,9,2}
$$

$$
\vdots
$$

$$
Is_{1,1,9} \Rightarrow \neg Is_{1,2,9}, \quad Is_{1,1,9} \Rightarrow \neg Is_{1,3,9}, \quad \ldots \quad Is_{1,1,9} \Rightarrow \neg Is_{1,9,9}
$$

This repeats for every row and value combination.

Column uniqueness - if a cell has value $v$, no other cell in its column can have $v$.

$$
Is_{1,1,1} \Rightarrow \neg Is_{2,1,1}, \quad Is_{1,1,1} \Rightarrow \neg Is_{3,1,1}, \quad \ldots \quad Is_{1,1,1} \Rightarrow \neg Is_{9,1,1}
$$

$$
Is_{1,1,2} \Rightarrow \neg Is_{2,1,2}, \quad Is_{1,1,2} \Rightarrow \neg Is_{3,1,2}, \quad \ldots \quad Is_{1,1,2} \Rightarrow \neg Is_{9,1,2}
$$

$$
\vdots
$$

$$
Is_{1,1,9} \Rightarrow \neg Is_{2,1,9}, \quad Is_{1,1,9} \Rightarrow \neg Is_{3,1,9}, \quad \ldots \quad Is_{1,1,9} \Rightarrow \neg Is_{9,1,9}
$$

This repeats for every column and value combination.

Box uniqueness - if a cell has value $v$, no other cell in its box can have $v$.

$$
Is_{1,1,1} \Rightarrow \neg Is_{2,2,1} \quad \ldots \quad \text{with every cell within its box}
$$

This repeats for every box and value combination.


#### 4. Givens

Each given is added as a fact on its own. E.g. if (2,3) is given as 7, we add $Is_{2,3,7}$.

For puzzle 1 this comes to 11,775 clauses over 729 symbols. That's 81 clauses for rule 1 and one clause for each of the 30 givens. Rule 2 and the row, column and box rules in rule 3 give 2,916 two-literal clauses each. So 81 + 4 x 2,916 + 30 = 11,775. This is the KB we used for resolution and model checking in 2.3(a).


### 1(b) Definite KB strategy

Here we need both symbol families. In a definite clause the premises and the conclusion all have to be positive symbols, so we can't use $\neg Is_{r,c,v}$:

- $Is_{r,c,v}$: cell $(r,c)$ has value $v$.
- $Not_{r,c,v}$: value $v$ is ruled out for cell $(r,c)$.

$Not_{r,c,v}$ is just a normal positive symbol, even though it's called "Not". It is not the same as $\neg Is_{r,c,v}$. That's what lets us write eliminations as definite clauses: a conjunction of positive premises and a single positive conclusion.

The rules are:

#### 1. Every cell has at least one value

The general version

$$
Is_{r,c,1} \lor Is_{r,c,2} \lor \cdots \lor Is_{r,c,n}
$$

has more than one positive literal, so it isn't a definite clause. `PropDefiniteKB.tell()` would reject it.

Instead we use "last candidate" rules. If every other value is ruled out for a cell, the cell must be $v$:

$$
\left(
\bigwedge_{u \ne v} Not_{r,c,u}
\right)
\Rightarrow Is_{r,c,v}
$$

E.g. for cell (1,1) and value 1 in a 9x9 puzzle:

$$
Not_{1,1,2} \land Not_{1,1,3} \land \cdots \land Not_{1,1,9}
\Rightarrow Is_{1,1,1}
$$

We add one of these for every value of every empty cell. This is weaker than the general clause. It only captures one consequence of "at least one value": it fires once all the other values are ruled out.

#### 2. Every cell has at most one value

If a cell is $v$, every other value $u$ is ruled out for that cell:

$$
Is_{r,c,v} \Rightarrow Not_{r,c,u}
\qquad \text{for every } u \ne v
$$

E.g. $Is_{1,1,1} \Rightarrow Not_{1,1,2}$. We add one implication per other value, so each rule still has a single positive conclusion.

#### 3. No two cells in the same row hold the same value

If cell $(r,c)$ is $v$, then $v$ is ruled out for every other cell in row $r$:

$$
Is_{r,c,v} \Rightarrow Not_{r,c',v}
\qquad \text{for every } c' \ne c
$$

#### 4. No two cells in the same column hold the same value

In the same way, $v$ is ruled out for every other cell in column $c$:

$$
Is_{r,c,v} \Rightarrow Not_{r',c,v}
\qquad \text{for every } r' \ne r
$$

#### 5. No two cells in the same box hold the same value

$v$ is also ruled out for every other cell in the same box:

$$
Is_{r,c,v} \Rightarrow Not_{r',c',v}
$$

where $(r',c')$ is a different cell in the same box. For the box we skip cells in the same row or column, since rules 3 and 4 already cover them.

#### 6. Given cells hold their stated values

Each given is added as a fact. E.g. if (1,1) is given as 5, we add:

$$
Is_{1,1,5}
$$

These facts are where inference starts.

#### Extra last-location rules (hidden singles)

We also add the same idea from the value's side. If 5 is ruled out for every other cell in a row, the last cell must be 5:

$$
\left(
\bigwedge_{c' \ne c} Not_{r,c',5}
\right)
\Rightarrow Is_{r,c,5}
$$

Same for columns and boxes. In Sudoku terms these are "hidden singles".

So the rules go back and forth: values give eliminations, eliminations give new values, and those values give more eliminations. Both FC and BC can use this cycle. That was enough to solve all 5 puzzles. A puzzle with fewer givens could still get stuck, since these rules don't cover every Sudoku technique (see Q4).

`build_definite_kb` also calls `kb_cleanup` for each given. It applies the given straight away, adding the $Not$ facts it implies and dropping rules that can never fire. This doesn't change what can be derived. It just makes the KB smaller: 14,484 clauses instead of 23,088 for puzzle 1.


### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**

No, we would not use either one as the default for Sudoku. Both are sound and complete. Sound means they never infer anything the KB doesn't entail. Complete means they can eventually derive everything it does entail. So in theory they always get the right answer. But these guarantees are only about the answer. They say nothing about how long "eventually" takes, or how much memory it needs. On a real 9x9 puzzle both take far too long to be useful. In our experiment above, neither of them could answer even one cell.

Model checking (`tt_entails`) is the brute-force method from the lecture. It tries every possible true/false assignment of the symbols. The query is entailed if it's true in every model where the KB is true. The number of models is $2^n$ for $n$ symbols, so the cost is exponential in the number of symbols. Our general KB has 729 symbols (one for each cell and value), so that is $2^{729}$ models, about $10^{219}$. For scale, there are only about $10^{80}$ atoms in the observable universe. A faster computer would not fix this, because every extra symbol doubles the work. For model checking the problem is mainly time.

Resolution (`pl_resolution`) is proof by contradiction. It adds the negated query to the KB and keeps resolving pairs of clauses until it gets the empty clause. Each round compares every pair of clauses, so one round is $O(N^2)$ for $N$ clauses. Every new resolvent is added back to the set of clauses, so $N$ grows after each round and the next round costs even more. Our KB starts with 11,775 clauses, which is already about 69 million pairs in the first round. The library also builds this whole list of pairs in memory before resolving anything. So for resolution the problem is space as well as time. In our run it used almost 1 GB within a few seconds and never finished the first round. Clause length makes it worse too. Resolving two long clauses usually gives another long clause, and longer clauses can resolve with more partners.

Size is the real issue for Sudoku, and it grows fast with the board. The number of symbols grows with the cube of the board size (729 for 9x9), and the number of clauses grows with its fourth power. Both algorithms are also fully general. They make no use of the fact that almost every Sudoku rule is a simple elimination.

We also tried both on a small 2x2 board. There they got the right answer in under a second, and resolution correctly said False for a wrong value. So the KB itself is correct and the issue is just the size of the problem. Even a 4x4 board (412 clauses) was already too big for both.

For Sudoku it makes more sense to use Horn clauses with forward or backward chaining. These only use Modus Ponens, which the lecture calls efficient inferencing. Forward chaining can run in time linear in the size of the KB, so there's no exponential blow-up. Our code is slower than that, because it scans the clause list to find the rules for each new fact, but it's still polynomial in the KB size, nowhere near $2^{729}$. The downside is that we can't write general disjunctions directly, so the encoding takes more work (see Question 1(b)). In return we get a solver that actually finishes. Resolution is still the right tool when we really need general clauses, or when we need to show that something is unsatisfiable. Chaining over a definite KB can't do that at all, because a definite KB is always satisfiable. It just shouldn't be the default at this size.

### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**
I started designing the algorithm based on the pseudocode provided in the lecture slides. Soon it was clear that was not enough. I ran into recursive loop problems and time complexity issues. To fix that, 3 main things were added on top of the provided lecture implementation.
1. `visited`: This set helps to check if a current cell has already been seen in the current recursive stack of checks (so that new recursive stacks can be reevaluated given new information), and if it has, it is not checked again (to prevent recursive loops if 2 variables are codependent).
2. `failed`: This is purely a time saving set, where every query which is bound to fail (not go anywhere meaningful) is added to this set. It makes the search much much more efficient. The whole set is cleared every time a new fact is added to the kb, since past routes that don't go anywhere might have an answer now.
3. `kb_cleanup`: This function removes redundant knowledge from the kb, providing efficiency to the solver. Once we know either `Is_r_c_v` or `Not_r_c_v`, any clause that uses its opposite can never fire, so it is removed. This helps massively after the first pass or first few passes where many more facts are known.

Each single call to `pl_bc_entails` stops because of `visited`. A symbol can't come up twice on the same recursion path, and there are only finitely many symbols, so every path has to end. For ensuring termination on every puzzle, `pl_bc_entails` was not modified but rather the solver itself. Looping through every cell is done inside a while loop with a state variable to check for changes to the kb. The puzzle is hence terminated if fully solved, or no updates to the kb are made while looping through the unsolved cells of the sudoku puzzle. This is because (1,1) might not be solvable because of some circular dependency with a later cell. In cases of circular dependency, the failed set marks this route as failed. Hence the unsolved cells would need to be iterated through again once that later cell is solved. This is due to the added optimizations of visited and failed that short circuit some routes on purpose. However, doing it this way ends up being much faster as more and more facts get added to the KB. It is to be noted however that most of the time, this while loop is not activated, rather it is more of a failsafe.

The base algorithm follows recursive function structure where it has a termination and propagation step. Several extra terminations are added on top of the base lecture slides using the 2 sets described above. There is also an added check for the `~q` clause as in our definite KB we use the Not keyword which is a positive literal. This step links the Not to the Is clause. For the propagation, every antecedent where `q` is the consequent is recursed through till we find an antecedent which we know to be true or false. Here, we slightly change the lecture slides function as we only add `q` to the kb rather than add every `p`. However, it works the same way as if `p` were to return true, it would have already been added to the kb before returning true. We also do not add every False return to the kb as false returns can be a result of short circuiting by the 2 sets added.

---
### Modified BC Pseudocode
For the purpose of this pseudocode ~q refers to the complementarty Is/Not variable, despite both technically being positive literals
```text
function pl_bc_entails?(KB,q,visited=None,failed=None) returns true or false
    inputs: KB, the knowledge base, a set of propositional definite clauses
            q: the query, a propositional symbol

At the start of each top-level query, initialize visited = {} and failed = {}.
Pass these same sets to every recursive call.
if q matches a fact, then return true
if q is already in the current recursion stack (visited) or in the failed set, then return False
if ~q is known in the kb, then return False and add q to failed set
add q to visited set
if there is no clause with a consequent that matches q, then remove q from visited set, add q to Failed set and return False

for each clause c in KB where q is in c.CONCLUSION do
    count = Number of symbols in c.PREMISE
    for all symbols p in c.PREMISE do
        if pl_bc_entails?(KB,p, visited, failed)
            count -= 1
        else break
    if count == 0
        if q not in kb
            add q to kb
            kb_cleanup(kb,q)
            clear the failed set
        remove q from visited
        return true
remove q from visited
add q to failed
return false
```


### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**

We'll use Naked Pairs. If two cells in the same row, column or box can only be $a$ or $b$, those two cells will take $a$ and $b$ between them. We may not know yet which cell gets which. Either way, no other cell in that unit can be $a$ or $b$.

E.g. say cells (1,1) and (1,4) can only be 2 or 7. They share a row, so they can't both hold the same value. One of them is 2 and the other is 7, so 2 and 7 can be ruled out for every other cell in row 1.

#### Encoding it

We can do this with the same symbols as `build_definite_kb`:

- $Is_{r,c,v}$ means cell $(r,c)$ has value $v$.
- $Not_{r,c,v}$ means value $v$ has been ruled out for cell $(r,c)$.

"Cell $(r,c)$ can only be $a$ or $b$" just means every other value is already ruled out for it. So we need an explicit $Not$ fact for every value outside $\{a,b\}$. For two cells $(r,c_1)$ and $(r,c_2)$ in the same row, the premise is:

$$
\bigwedge_{u \notin \{a,b\}} \left( Not_{r,c_1,u} \land Not_{r,c_2,u} \right)
$$

Together these premises say both cells are down to the same two values. They only use $Not$ facts that were actually derived. We never treat "no $Is$ fact yet" as meaning a value is impossible.

Then for every other column $c_3 \notin \{c_1,c_2\}$ in row $r$, we add two rules:

$$
\left[ \bigwedge_{u \notin \{a,b\}} \left( Not_{r,c_1,u} \land Not_{r,c_2,u} \right) \right] \Rightarrow Not_{r,c_3,a}
$$

$$
\left[ \bigwedge_{u \notin \{a,b\}} \left( Not_{r,c_1,u} \land Not_{r,c_2,u} \right) \right] \Rightarrow Not_{r,c_3,b}
$$

All the premises are positive atoms and there's one positive conclusion, so these are definite clauses. The $Not$ symbols are ordinary positive atoms here, the same as in 1(b). They aren't logical negations. Columns and boxes work the same way. There the conclusions rule $a$ and $b$ out for the other cells in that column or box.

#### Example

Say cells (1,1) and (1,4) are down to $\{2,7\}$ in a 9x9 puzzle. The KB must already know that both cells exclude every value in

$$
E = \{1,3,4,5,6,8,9\}
$$

$E$ is just shorthand for a set of digits. It isn't an extra propositional symbol. For another cell in the row, like (1,6), we add:

$$
\left[ \bigwedge_{u \in E} \left( Not_{1,1,u} \land Not_{1,4,u} \right) \right] \Rightarrow Not_{1,6,2}
$$

$$
\left[ \bigwedge_{u \in E} \left( Not_{1,1,u} \land Not_{1,4,u} \right) \right] \Rightarrow Not_{1,6,7}
$$

Each rule has 14 premises: seven ruled-out values for each of the two cells. We add the same rules for every other cell in row 1. Once the premises are known, FC can derive the new eliminations. These can then trigger the existing last-candidate rules and fill in more cells.

#### What happens if we add them

- It's stronger. FC can now rule values out even when neither cell's value is known yet, so it can make progress where the current rules get stuck.
- The KB gets a lot bigger. We need rules for every pair of cells, every pair of values and every other cell, in every row, column and box. There are 27 units, 36 pairs of cells in each, 36 pairs of values and 7 other cells, with 2 rules each. That's about 490,000 new rules with 14 premises each, more than 20 times the size of our current KB. Generating all of them up front uses a lot of memory.
- It costs time. The new rules may solve more cells, but FC and BC both have more rules to go through, and most of these will never fire for a given puzzle. Indexing rules by their premises would cut down the scanning in FC.
- It's still not complete. Some puzzles need other techniques like X-Wing, or guessing. FC stays complete for the definite KB, so it derives every fact these rules imply. For some puzzles that still leaves empty cells.

So definite clauses can express more than single-cell deductions, and Naked Pairs fit in Horn logic fine. The catch is that every technique has to be written out ahead of time. Each one adds a lot of rules that we have to store and apply.

#### Why the definite KB can still get stuck

FC and BC are complete for a definite KB, so they find every atom the KB entails. The problem is that our rules are weaker than the real Sudoku constraints. They don't keep all the logical consequences of those constraints. The general clause

$$
Is_{r,c,1} \lor Is_{r,c,2} \lor \cdots \lor Is_{r,c,n}
$$

becomes a set of last-candidate rules, one for each value $v$:

$$
\left( \bigwedge_{u \ne v} Not_{r,c,u} \right) \Rightarrow Is_{r,c,v}
$$

These let us infer a value once all $n-1$ others are ruled out. They aren't logically equivalent to the disjunction, though. When two or more candidates are left, nothing forces one of them to be true. Also, each $Not_{r,c,u}$ is its own atom for an elimination we've derived. The KB doesn't know that it means $\neg Is_{r,c,u}$.

So FC can reach a fixed point, where nothing new can be derived, while some cells are still empty. BC is complete over the same KB, and it can't prove those cells either. Getting further needs more rules like Naked Pairs, or backtracking: try a value, follow what it implies, and undo it if it leads to a contradiction. So the limit comes from the rules we encode. The chaining itself is still complete.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Your answer:**

#### 5(a) When is backward chaining faster than forward chaining

BC is faster when the query only depends on a small part of a big KB, or when most of the symbols in that part are already known facts. BC starts from the query and only looks at rules for the symbols the query actually needs. When one of those symbols is already a fact, there's no recursion needed to work out its value, so BC stops almost straight away. FC doesn't know what we're asking for. It starts from all the known facts and keeps going through the clauses, deriving new facts until the query happens to come up. So it does a lot of work that has nothing to do with the query.

The first experiment below shows this. We made a puzzle where only the target cell's row and column are filled in, so the answer depends on very little of the KB. BC answered the query noticeably faster than FC on the same puzzle.

#### 5(b) When backward chaining performs worse than forward chaining

Solving the full grid is the opposite case. We ask about all 81 cells, one after another. Each query's relevant part might be small, but BC keeps redoing the same recursive searches from one query to the next. Our solver cuts some of this down. It adds proven facts back into the KB, and `kb_cleanup` clears out the failed values of a solved cell along with the rules that can't fire any more. It doesn't get rid of the repeated searches completely though. FC is better for the full grid, especially when it also feeds the facts it derives back into the KB, because each pass derives lots of facts at once.

That's why we also wrote `solve_full_grid_fc_one_pass`. It runs forward chaining once and keeps going until every cell has a value, instead of starting again for each cell. This removes all the repeated work in `solve_full_grid_fc`.

The second experiment below times all three on the 5 puzzles. BC is slower than both FC versions on all of them. BC's time goes up the most as the number of givens goes down, so the gap is biggest on puzzle 1, which has the fewest givens. Each time also includes building the definite KB, which is the same work for all three, so the differences come from the chaining itself.

The benchmark below is one run, but we ran the whole notebook three times on two laptops. The times moved between runs, but every time BC's time grew the most as the givens went down, and the gap was biggest on puzzle 1.

Potential improvements for backward chaining - Instead of maintaining the clauses within KB as a list, they can be indexed by consequents. The look up time will then reduce, as the antecedents for every query can be obtained in constant time.

In [10]:
# Experiment to prove backward chaining can be faster than forward chaining
from IPython.display import Markdown, display

sample_puzzle = puzzle_pool[0]
sample_solution = sample_puzzle['solution']
query_cell = (5, 5)
query_value = sample_solution[query_cell]

# Populate every other cell in the target row and target column.
sample_givens = {}
for c in range(1, n + 1):
    if c != query_cell[1]:
        sample_givens[(query_cell[0], c)] = sample_solution[(query_cell[0], c)]

for r in range(1, n + 1):
    if r != query_cell[0]:
        sample_givens[(r, query_cell[1])] = sample_solution[(r, query_cell[1])]

assert query_cell not in sample_givens
assert len(sample_givens) == (2 * n) - 2
query = atom('Is', query_cell[0], query_cell[1], query_value)

print(f'Synthetic puzzle: {len(sample_givens)} givens; target {query_cell} is empty')
for r in range(1, n + 1):
    print([str(sample_givens.get((r, c), '.')) for c in range(1, n + 1)])

# Use separate but equivalent KBs. KB construction is intentionally not timed.
single_query_fc_kb = build_definite_kb(n, box_h, box_w, sample_givens)
single_query_bc_kb = build_definite_kb(n, box_h, box_w, sample_givens)

start_time = time.perf_counter()
fc_result = pl_fc_entails(single_query_fc_kb, query)
fc_query_time = time.perf_counter() - start_time
assert fc_result is True

start_time = time.perf_counter()
bc_result = pl_bc_entails(single_query_bc_kb, query)
bc_query_time = time.perf_counter() - start_time
assert bc_result is True

single_query_table = [
    '| Measurement | Result |',
    '|---|---:|',
    f'| Number of givens | {len(sample_givens)} |',
    f'| Query | `{query}` |',
    f'| Total KB clauses | {len(single_query_fc_kb.clauses):,} |',
    f'| FC entailment time | {fc_query_time:.4f} s |',
    f'| BC entailment time | {bc_query_time:.4f} s |',
    f'| BC speedup over FC | {fc_query_time / bc_query_time:.2f}x |',
]

display(Markdown('\n'.join(single_query_table)))

Synthetic puzzle: 16 givens; target (5, 5) is empty
['.', '.', '.', '.', '8', '.', '.', '.', '.']
['.', '.', '.', '.', '1', '.', '.', '.', '.']
['.', '.', '.', '.', '6', '.', '.', '.', '.']
['.', '.', '.', '.', '9', '.', '.', '.', '.']
['3', '6', '9', '8', '.', '2', '5', '1', '4']
['.', '.', '.', '.', '3', '.', '.', '.', '.']
['.', '.', '.', '.', '4', '.', '.', '.', '.']
['.', '.', '.', '.', '5', '.', '.', '.', '.']
['.', '.', '.', '.', '2', '.', '.', '.', '.']


| Measurement | Result |
|---|---:|
| Number of givens | 16 |
| Query | `Is5_5_7` |
| Total KB clauses | 18,652 |
| FC entailment time | 2.0289 s |
| BC entailment time | 0.4875 s |
| BC speedup over FC | 4.16x |

In [11]:
# Experiment to check performance of all 3 algorithms on all given puzzles
from IPython.display import Markdown, display

algorithms = [
    ('solve_full_grid_fc_one_pass', solve_full_grid_fc_one_pass),
    ('solve_full_grid_bc', solve_full_grid_bc),
    ('solve_full_grid_fc', solve_full_grid_fc),
]

assert len(puzzle_pool) == 5, f'Expected 5 puzzles, found {len(puzzle_pool)}'
benchmark_rows = []

for puzzle_number, current_puzzle in enumerate(puzzle_pool, start=1):
    timings = {}
    answers = {}

    for algorithm_name, solver in algorithms:
        start_time = time.perf_counter()
        solved = solver(n, box_h, box_w, dict(current_puzzle['givens']))
        elapsed_time = time.perf_counter() - start_time

        assert solved == current_puzzle['solution'], (
            f'Puzzle {puzzle_number}: {algorithm_name} returned an incorrect solution'
        )

        timings[algorithm_name] = elapsed_time
        answers[algorithm_name] = solved

    assert (
        answers['solve_full_grid_fc_one_pass']
        == answers['solve_full_grid_bc']
        == answers['solve_full_grid_fc']
    ), f'Puzzle {puzzle_number}: the three algorithms returned different answers'

    benchmark_rows.append({
        'Puzzle': puzzle_number,
        'Given cells': current_puzzle['given_count'],
        'solve_full_grid_fc_one_pass (s)': timings['solve_full_grid_fc_one_pass'],
        'solve_full_grid_bc (s)': timings['solve_full_grid_bc'],
        'solve_full_grid_fc (s)': timings['solve_full_grid_fc'],
    })

    print(f'Puzzle {puzzle_number}: all three solutions verified')

table_lines = [
    '| Puzzle | Given cells | FC one pass (s) | BC (s) | FC (s) |',
    '|---:|---:|---:|---:|---:|',
]

for row in benchmark_rows:
    table_lines.append(
        f"| {row['Puzzle']} | {row['Given cells']} "
        f"| {row['solve_full_grid_fc_one_pass (s)']:.4f} "
        f"| {row['solve_full_grid_bc (s)']:.4f} "
        f"| {row['solve_full_grid_fc (s)']:.4f} |"
    )

display(Markdown('\n'.join(table_lines)))

Puzzle 1: all three solutions verified


Puzzle 2: all three solutions verified


Puzzle 3: all three solutions verified


Puzzle 4: all three solutions verified


Puzzle 5: all three solutions verified


| Puzzle | Given cells | FC one pass (s) | BC (s) | FC (s) |
|---:|---:|---:|---:|---:|
| 1 | 30 | 21.9527 | 63.6736 | 35.2070 |
| 2 | 33 | 21.0764 | 39.3935 | 25.6196 |
| 3 | 36 | 23.1598 | 34.7399 | 26.6057 |
| 4 | 39 | 22.3700 | 28.5750 | 21.4097 |
| 5 | 42 | 19.8615 | 22.1722 | 20.7300 |

## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

[https://it5005-g1-sudoku.streamlit.app](https://it5005-g1-sudoku.streamlit.app)

If nobody has opened the app for a while, Streamlit puts it to sleep. Click the button on that page to wake it up, which takes about a minute. The app runs on Streamlit's free server, so solving a whole grid takes 1 to 2 minutes (about 80 s for FC and 110 s for BC in our test). The first question about an empty cell also takes about a minute. Tutor mode answers straight away.
